In [10]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [12]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [13]:
# Завдання 5. Ноутбук 04 — снепшот поточних курсів

# Завдання 5.1. Прочитайте nbu_raw.raw_rates, розгорніть payload у колонки і нормалізуйте значення так само, як у завданні 3.


raw_rates = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""

query = client.query(raw_rates).to_dataframe()

payload = pd.json_normalize(query["payload"].map(json.loads))
payload["cc"] = payload["cc"].str.strip().str.upper()
payload["txt"] = payload["txt"].str.strip()
payload["r030"] = payload["r030"].astype("Int64")

payload = payload.rename(columns={
    "cc": "currency_code",
    "txt": "currency_name"
}) 


payload["business_date"] = query["business_date"].values
payload["ingested_at"] = query["ingested_at"].values

# Завдання 5.2. Для кожної валюти залиште рядок із максимальною business_date. 

snap = (payload.sort_values(["business_date", "ingested_at"])
           .drop_duplicates(subset="currency_code", keep="last"))


# Завдання 5.3. Приєднайте currency_key із таблиці nbu_dwh.dim_currency — не вигадуйте ключі заново. 
# Валютам, яких немає у вимірі, поставте -1.

dim_currency_bq = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
"""

dim_currency = client.query(dim_currency_bq).to_dataframe()

snap_dim_cur = snap.merge(
    dim_currency[["currency_code", "currency_key"]],
    on="currency_code",
    how="left"
)



snap_dim_cur["currency_key"] = (
    snap_dim_cur["currency_key"]
    .fillna(-1)
    .astype("Int64")
)

snap_dim_cur

/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,currency_code,exchangedate,r030,rate,special,currency_name,business_date,ingested_at,currency_key
0,DZD,28.08.2026,12,0.334910,None,Алжирський динар,2026-08-28,2026-08-28 12:10:12.672606,1
1,AUD,28.08.2026,36,32.006600,None,Австралійський долар,2026-08-28,2026-08-28 12:10:12.672641,2
2,BDT,28.08.2026,50,0.363280,None,Така,2026-08-28,2026-08-28 12:10:12.672653,3
3,CAD,28.08.2026,124,32.113200,None,Канадський долар,2026-08-28,2026-08-28 12:10:12.672664,4
4,CNY,28.08.2026,156,6.627500,None,Юань Женьміньбі,2026-08-28,2026-08-28 12:10:12.672674,5
5,CZK,28.08.2026,203,2.148000,None,Чеська крона,2026-08-28,2026-08-28 12:10:12.672684,6
6,DKK,28.08.2026,208,6.937900,None,Данська крона,2026-08-28,2026-08-28 12:10:12.672694,7
7,HKD,28.08.2026,344,5.682800,None,Гонконгівський долар,2026-08-28,2026-08-28 12:10:12.672707,8
8,HUF,28.08.2026,348,0.142439,None,Форинт,2026-08-28,2026-08-28 12:10:12.672717,9
9,INR,28.08.2026,356,0.466220,None,Індійська рупія,2026-08-28,2026-08-28 12:10:12.672727,10


In [14]:
# Завдання 5.4. Додайте колонку snapshot_ts — момент побудови снепшоту

snap_dim_cur["snapshot_ts"] = pd.Timestamp.now(tz="UTC")


snap_dim_cur = snap_dim_cur[[
    "currency_key", 
    "currency_code", 
    "currency_name", 
    "rate", 
    "business_date", 
    "snapshot_ts"
]]

snap_dim_cur

,currency_key,currency_code,currency_name,rate,business_date,snapshot_ts
0,1,DZD,Алжирський динар,0.334910,2026-08-28,2026-08-28 16:49:37.920531+00:00
1,2,AUD,Австралійський долар,32.006600,2026-08-28,2026-08-28 16:49:37.920531+00:00
2,3,BDT,Така,0.363280,2026-08-28,2026-08-28 16:49:37.920531+00:00
3,4,CAD,Канадський долар,32.113200,2026-08-28,2026-08-28 16:49:37.920531+00:00
4,5,CNY,Юань Женьміньбі,6.627500,2026-08-28,2026-08-28 16:49:37.920531+00:00
5,6,CZK,Чеська крона,2.148000,2026-08-28,2026-08-28 16:49:37.920531+00:00
6,7,DKK,Данська крона,6.937900,2026-08-28,2026-08-28 16:49:37.920531+00:00
7,8,HKD,Гонконгівський долар,5.682800,2026-08-28,2026-08-28 16:49:37.920531+00:00
8,9,HUF,Форинт,0.142439,2026-08-28,2026-08-28 16:49:37.920531+00:00
9,10,INR,Індійська рупія,0.466220,2026-08-28,2026-08-28 16:49:37.920531+00:00


In [15]:

# Завдання 5.5. Запишіть у nbu_dwh.snapshot_rates_current

snap_path = f"{PROJECT_ID}.nbu_dwh.snapshot_rates_current"

schema = [
    bigquery.SchemaField("currency_key", "INT64"),
    bigquery.SchemaField("currency_code", "STRING"),
    bigquery.SchemaField("currency_name", "STRING"),
    bigquery.SchemaField("rate", "FLOAT64"),
    bigquery.SchemaField("business_date", "DATE"),
    bigquery.SchemaField("snapshot_ts", "TIMESTAMP"),

]

cfg = bigquery.LoadJobConfig(schema=schema,
                             write_disposition="WRITE_TRUNCATE")

client.load_table_from_dataframe(snap_dim_cur, snap_path, job_config=cfg).result()

print("The snapshot data added to BigQuery successfully!")







The snapshot data added to BigQuery successfully!


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [53]:
# Завдання 5.6. Перевірте й виведіть результат:


print(f"""Correct numbers of rows: {len(snap_dim_cur) == payload['currency_code'].nunique()}
In bronze layer are {payload['currency_code'].nunique()} unique currencies and the snapshot table has {len(snap_dim_cur)} rows""")
print(f"All currency_code values are unique: {snap_dim_cur['currency_code'].is_unique}")
print(f"Is there any currency_key = -1 values in the table: {(snap_dim_cur['currency_key'] == -1).any()}")



Correct numbers of rows: True
In bronze layer are 45 unique currencies and the snapshot table has 45 rows
All currency_code values are unique: True
Is there any currency_key = -1 values in the table: False
